In [ ]:
"""
====================================================================================
Ethereum Transaction Logs Preprocessing and Cleaning
====================================================================================

Purpose
-------
This module prepares Ethereum transaction receipt logs for token transfer decoding
and analysis. It ensures that only valid Transfer events are retained and that
all hexadecimal fields are standardized.

Workflow
--------
1. Safely parse the 'logs' column from string representation to Python lists.
2. Remove rows with empty logs.
3. Filter each row's logs to retain only Transfer events (identified by the standard
   ERC-20 Transfer event topic hash).
4. Keep only rows that contain at least two Transfer logs (common for swaps).
5. Clean each log:
   - Standardize the address to lowercase.
   - Fix malformed 'data' hex values.
   - Standardize topics (sender and recipient addresses are 42-character hex).
6. Remove any unnamed columns generated during DataFrame operations.

Inputs
------
- `extracted_block_receipts`: a DataFrame containing Ethereum block receipts with a 
  'logs' column (raw event logs).

Outputs
-------
- `df_filtered`: a cleaned DataFrame containing only valid Transfer events with
  corrected hexadecimal values, ready for token decoding and further analysis.

Use Case
--------
- This preprocessing step is essential for detecting token transfers, swaps, and
  sandwich attack patterns in DeFi datasets.
- Ensures robustness against malformed data from blockchain event logs.
"""


In [ ]:
import pandas as pd
import ast

TRANSFER_TOPIC = "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"

df = extracted_block_receipts.copy()

def parse_logs(logs):
    if isinstance(logs, str):
        try:
            return ast.literal_eval(logs)
        except Exception:
            return []
    elif isinstance(logs, list):
        return logs
    return []

df['logs'] = df['logs'].apply(parse_logs)
df_filtered = df[df['logs'].apply(lambda logs: len(logs) > 0)]

def filter_transfer_logs(logs):
    if not isinstance(logs, list):
        return []
    return [
        log for log in logs
        if isinstance(log, dict) and len(log.get("topics", [])) > 0 and log["topics"][0].lower() == TRANSFER_TOPIC
    ]

df_filtered['logs'] = df_filtered['logs'].apply(filter_transfer_logs)
df_filtered = df_filtered[df_filtered['logs'].apply(lambda logs: len(logs) >= 2)]

def clean_logs(logs):
    def fix_hex_value(value):
        if not value:
            return "0x0"
        value_str = str(value).strip()
        if value_str == "0x" or not value_str:
            return "0x0"
        if not value_str.startswith("0x"):
            value_str = "0x" + value_str
        if value_str == "0x" or len(value_str.strip()) <= 2:
            return "0x0"
        return value_str
    
    cleaned = []
    for log in logs:
        try:
            log_address = log["address"].lower()
            log_data = fix_hex_value(log.get("data", ""))
            topics = []
            for i, topic in enumerate(log["topics"]):
                topic = fix_hex_value(str(topic).lower())
                if i in [1, 2]:
                    if len(topic) >= 42:
                        topic = "0x" + topic[-40:]
                    else:
                        topic_without_0x = topic[2:] if topic.startswith('0x') else topic
                        topic = "0x" + topic_without_0x.zfill(40)
                topics.append(topic)
            cleaned.append({
                **log,
                "address": log_address,
                "topics": topics,
                "data": log_data
            })
        except Exception:
            continue
    return cleaned

df_filtered['logs'] = df_filtered['logs'].apply(clean_logs)
df_filtered = df_filtered.loc[:, ~df_filtered.columns.str.contains('^Unnamed')]


In [ ]:
"""
====================================================================================
Ethereum Token Transfer Extraction with Intermediary Detection
====================================================================================

Purpose
-------
This module decodes Ethereum transaction logs to extract the input and output tokens
(token_in and token_out) and their respective values. The logic accounts for direct
transfers as well as transfers involving intermediary addresses, which is common
in multi-step DeFi swaps and sandwich attacks.

Workflow
--------
1. Convert hexadecimal values from log 'data' fields safely to integers.
2. Detect token transfers:
   - Direct transfers from the pool to the target address.
   - Transfers through intermediary addresses up to a specified depth.
3. Normalize Ethereum addresses and hex data for consistency.
4. Apply extraction logic to each transaction row in a DataFrame, handling malformed
   or missing logs safely.
5. Aggregate results into four new columns:
   - `token_in`: input token address
   - `value_token_in`: input token value
   - `token_out`: output token address
   - `value_token_out`: output token value

Inputs
------
- `df`: DataFrame containing transaction data with a 'logs' column (preprocessed and
  cleaned).
- `max_depth`: Maximum depth of intermediary search for transfers (default=2).

Outputs
-------
- Original DataFrame with four additional columns: `token_in`, `value_token_in`,
  `token_out`, and `value_token_out`.

Use Case
--------
- This extraction is critical for DeFi analysis, including:
  - Detecting sandwich attacks
  - Multi-hop swap analysis
  - Token flow tracking in Ethereum transactions
- Robust to incomplete, malformed, or nested log structures.
"""


In [ ]:
import pandas as pd
import ast

def safe_hex_to_int(x):
    if not x or x == '0x' or x == '':
        return 0
    try:
        hex_str = str(x).strip()
        if hex_str.startswith('0x'):
            hex_str = hex_str[2:]
        if not hex_str:
            return 0
        return int(hex_str, 16)
    except (ValueError, TypeError):
        return 0

def look_for_intermediaries_no_prints(current_recipient, max_depth, transfer_logs, pool_address, token_in):
    for depth in range(1, max_depth + 1):
        intermediary_transfers = []
        for idx, log in enumerate(transfer_logs):
            topic1 = log["topics"][1]
            topic2 = log["topics"][2]
            address = log["address"]
            value = safe_hex_to_int(log["data"])
            if topic2 == current_recipient:
                intermediary_transfers.append({
                    'idx': idx,
                    'sender': topic1,
                    'recipient': topic2,
                    'token': address,
                    'value': value
                })
        if not intermediary_transfers:
            return None, 0
        for transfer in intermediary_transfers:
            intermediary_address = transfer['sender']
            potential_token_out = transfer['token']
            for idx, log in enumerate(transfer_logs):
                topic1 = log["topics"][1]
                topic2 = log["topics"][2]
                address = log["address"]
                value = safe_hex_to_int(log["data"])
                if topic1 == pool_address and topic2 == intermediary_address:
                    return potential_token_out, value
    return None, 0

def find_token_out_with_intermediaries_no_prints(tx_from, potential_tx_from, token_in, pool_address, transfer_logs, max_depth=2):
    if not transfer_logs:
        return None, 0
    tx_from = tx_from
    potential_tx_from = potential_tx_from if potential_tx_from else None
    pool_address = pool_address
    token_in = token_in if token_in else None
    for log in transfer_logs:
        topic1 = log["topics"][1]
        topic2 = log["topics"][2]
        address = log["address"]
        value = safe_hex_to_int(log["data"])
        if topic1 == pool_address and topic2 == tx_from:
            return address, value
    for log in transfer_logs:
        topic1 = log["topics"][1]
        topic2 = log["topics"][2]
        address = log["address"]
        value = safe_hex_to_int(log["data"])
        if potential_tx_from and topic1 == pool_address and topic2 == potential_tx_from:
            return address, value
    if max_depth == 0:
        return None, 0
    token_out, value = look_for_intermediaries_no_prints(tx_from, max_depth, transfer_logs, pool_address, token_in)
    if token_out:
        return token_out, value
    token_out, value = look_for_intermediaries_no_prints(potential_tx_from, max_depth, transfer_logs, pool_address, token_in)
    if token_out:
        return token_out, value
    current_pool_address = pool_address
    current_token_out = None
    current_value = 0
    potential_token_out = None
    for log in transfer_logs:
        topic1 = log["topics"][1]
        topic2 = log["topics"][2]
        address = log["address"]
        value = safe_hex_to_int(log["data"])
        if topic1 == current_pool_address:
            current_token_out = address
            current_value = value
            current_pool_address = topic2
            if address != token_in:
                potential_token_out = address
    if current_token_out == token_in:
        current_token_out = potential_token_out
    if current_token_out:
        return current_token_out, current_value
    return None, 0

def extract_token_in_out_values_with_intermediaries_no_prints(tx_from, transfer_logs, max_depth=2):
    token_in = None
    pool_address = None
    tx_from_normalized = tx_from
    potential_tx_from_normalized = None
    found_token_in = False
    for idx, log in enumerate(transfer_logs):
        topic1 = log["topics"][1]
        topic2 = log["topics"][2]
        address = log["address"]
        value = safe_hex_to_int(log["data"])
        if topic1 == tx_from_normalized:
            token_in = address
            value_token_in = value
            pool_address = topic2
            found_token_in = True
            break
    if not found_token_in:
        first_log = transfer_logs[0]
        token_in = first_log["address"]
        value_token_in = int(first_log["data"], 16) if first_log["data"] else 0
        pool_address = first_log["topics"][2]
        potential_tx_from_normalized = first_log["topics"][1]
    if pool_address:
        token_out, value_token_out = find_token_out_with_intermediaries_no_prints(
            tx_from_normalized, potential_tx_from_normalized, token_in, pool_address, transfer_logs, max_depth
        )
    else:
        token_out = None
        value_token_out = 0
    return token_in, value_token_in, token_out, value_token_out

def apply_token_extraction_to_df_no_prints(df, max_depth=2):
    def safe_parse_logs(log_field):
        if isinstance(log_field, list):
            return log_field
        if isinstance(log_field, str):
            try:
                parsed = ast.literal_eval(log_field)
                if isinstance(parsed, list):
                    return parsed
            except (ValueError, SyntaxError):
                return []
        return []
    def extract_row(row):
        try:
            parsed_logs = safe_parse_logs(row['logs'])
            result = extract_token_in_out_values_with_intermediaries_no_prints(
                tx_from=row['from'],
                transfer_logs=parsed_logs,
                max_depth=max_depth,
            )
            if not isinstance(result, (list, tuple)) or len(result) != 4:
                return (None, 0, None, 0)
            token_in, value_in, token_out, value_out = result
            if token_out is None or value_out is None:
                token_out, value_out = None, 0
            if token_in is None or value_in is None:
                token_in, value_in = None, 0
            return token_in, value_in, token_out, value_out
        except Exception:
            return (None, 0, None, 0)
    return pd.concat([df], axis=1)


cleaned_transactions_df = apply_token_extraction_to_df_no_prints(df_filtered, max_depth=3)
valid_transactions = cleaned_transactions_df[~((cleaned_transactions_df['token_in'].isnull()) & (cleaned_transactions_df['token_out'].isnull()))]

In [ ]:
"""
====================================================================================
Merging Decoded Infura Data with Blocknative Transaction Data
====================================================================================

Purpose
-------
This module merges Ethereum transaction data decoded from Infura with preprocessed
Blocknative data. The goal is to consolidate enriched transaction information, 
including token transfers and gas usage, while optimizing memory usage.

Workflow
--------
1. Standardize column names in the decoded Infura DataFrame to match merge keys.
2. Convert hexadecimal columns (block numbers, transaction indexes) to integers.
3. Select only relevant columns to minimize memory footprint.
4. Ensure consistent data types for merge keys (`transaction_hash` and `hash`).
5. Perform a left join between the Infura and Blocknative DataFrames on transaction
   hashes.
6. Drop unnecessary columns and any autogenerated index columns.

Inputs
------
- `enhanced_valid_df`: DataFrame containing decoded Infura transactions with token
  information and gas usage.
- `treated_blocknative_aout`: Preprocessed Blocknative DataFrame containing metadata
  for each transaction.

Outputs
-------
- Merged DataFrame containing:
  - Transaction order in block
  - Transaction hash
  - Token in/out and corresponding values
  - Gas usage and cumulative gas
  - Block number and additional metadata from Blocknative
"""

In [ ]:
import pandas as pd

valid_transactions.rename(columns={
    'transactionHash': 'transaction_hash',
    'gasUsed': 'gas_used',
    'cumulativeGasUsed': 'cumulative_gas_used',
    'blockNumber': 'block_number',
    'transactionIndex': 'tx_order_in_block'
}, inplace=True)

valid_transactions['block_number'] = valid_transactions['block_number'].apply(lambda x: int(x, 16) if isinstance(x, str) else x)
valid_transactions['tx_order_in_block'] = valid_transactions['tx_order_in_block'].apply(lambda x: int(x, 16) if isinstance(x, str) else x)

cols_to_keep = [
    'tx_order_in_block', 'transaction_hash',
    'token_in', 'token_out',
    'value_token_in', 'value_token_out',
    'cumulative_gas_used', 'block_number', 'gas_used'
]
valid_transactions_filtered = valid_transactions[cols_to_keep]

cleaned_blocknative_data = cleaned_blocknative_data.drop(columns=['input'])
valid_transactions_filtered['transaction_hash'] = valid_transactions_filtered['transaction_hash'].astype(str)
cleaned_blocknative_data['hash'] = cleaned_blocknative_data['hash'].astype(str)

df_merged = valid_transactions_filtered.merge(
    cleaned_blocknative_data,
    left_on='transaction_hash',
    right_on='hash',
    how='left',
    suffixes=('_valid_transactions', '_cleaned_blocknative_data')
)

df_merged.drop(columns=['hash'], inplace=True)
df_merged = df_merged.loc[:, ~df_merged.columns.str.contains('^Unnamed')]
